# Tagging and Extraction Using OpenAI functions

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_tool

In [3]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [4]:
convert_to_openai_tool(Tagging)

{'type': 'function',
 'function': {'name': 'Tagging',
  'description': 'Tag the piece of text with particular info.',
  'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, or `neutral`',
     'type': 'string'},
    'language': {'description': 'language of text (should be ISO 639-1 code)',
     'type': 'string'}},
   'required': ['sentiment', 'language'],
   'type': 'object'}}}

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [6]:
model = ChatOpenAI(temperature=0)

In [7]:
tagging_tools = [convert_to_openai_tool(Tagging)]

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [9]:
model_with_tools = model.bind_tools(
    [Tagging],
    tool_choice={"type": "function", "function": {"name": "Tagging"}}
)

In [10]:
tagging_chain = prompt | model_with_tools

In [11]:
tagging_chain.invoke({"input": "I love langchain"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 108, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbO39rQaovNn7Yy9bd3f6iYE17xO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7085-2eac-7941-b3fb-009b162fd5d3-0', tool_calls=[{'name': 'Tagging', 'args': {'sentiment': 'pos', 'language': 'en'}, 'id': 'call_xPUUC3n2ZKXpU0AYclErFLVP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 10, 'total_tokens': 118, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 111, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbO8gjqqkRHG2u0wWWVe20Kgja0Z', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7085-4421-7f83-b0be-dc5ad5674e57-0', tool_calls=[{'name': 'Tagging', 'args': {'sentiment': 'neg', 'language': 'it'}, 'id': 'call_FYecIhMBuqwUAlyikpC48kUJ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 10, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser

In [14]:
tagging_chain = prompt | model_with_tools | PydanticToolsParser(tools=[Tagging], first_tool_only=True)

In [15]:
tagging_chain.invoke({"input": "non mi piace questo cibo"})

Tagging(sentiment='neg', language='it')

## Extraction

Extraction is similar to tagging, but used for extracting multiple pieces of information.

In [16]:
from typing import Optional
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")

In [17]:
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [18]:
convert_to_openai_tool(Information)

{'type': 'function',
 'function': {'name': 'Information',
  'description': 'Information to extract.',
  'parameters': {'properties': {'people': {'description': 'List of info about people',
     'items': {'description': 'Information about a person.',
      'properties': {'name': {'description': "person's name",
        'type': 'string'},
       'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
        'description': "person's age"}},
      'required': ['name', 'age'],
      'type': 'object'},
     'type': 'array'}},
   'required': ['people'],
   'type': 'object'}}}

In [19]:
extraction_model = model.bind_tools(
    [Information],
    tool_choice={"type": "function", "function": {"name": "Information"}}
)

In [20]:
extraction_model.invoke("Joe is 30, his mom is Martha")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 95, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbOmH9i9XFvx19p2detFKLFX1qOZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7085-dda6-7601-8567-c067d0da825c-0', tool_calls=[{'name': 'Information', 'args': {'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': None}]}, 'id': 'call_89jHzNf1ewg237doVStliOMW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 21, 'total_tokens': 116, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [21]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

In [22]:
extraction_chain = prompt | extraction_model

In [23]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 112, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSbOsk8CI9HUs4QreW0nlupfnLpDc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7085-f6f4-7f21-a346-bbc6ef105949-0', tool_calls=[{'name': 'Information', 'args': {'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': None}]}, 'id': 'call_k4yo72pVaTOWYaV0iTFCPCeM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 21, 'total_tokens': 133, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio

In [24]:
extraction_chain = prompt | extraction_model | PydanticToolsParser(tools=[Information], first_tool_only=True)

In [25]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

Information(people=[Person(name='Joe', age=30), Person(name='Martha', age=None)])

In [ ]:
# PydanticToolsParser already imported above; extract nested key with a lambda

In [26]:
extraction_chain = prompt | extraction_model | PydanticToolsParser(tools=[Information], first_tool_only=True) | (lambda x: x.people)

In [27]:
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

[Person(name='Joe', age=30), Person(name='Martha', age=None)]

## Doing it for real

We can apply tagging to a larger body of text.

For example, let's load this blog post and extract tag information from a sub-set of the text.

In [28]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [29]:
doc = documents[0]

In [30]:
page_content = doc.page_content[:10000]

In [31]:
print(page_content[:1000])







LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.


In [32]:
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [33]:
tagging_model = model.bind_tools(
    [Overview],
    tool_choice={"type": "function", "function": {"name": "Overview"}}
)
tagging_chain = prompt | tagging_model | PydanticToolsParser(tools=[Overview], first_tool_only=True)

In [34]:
tagging_chain.invoke({"input": page_content})

Overview(summary='This article discusses the concept of building autonomous agents powered by LLM (large language model) as the core controller. It covers components like planning, memory, and tool use, along with case studies and challenges. The focus is on task decomposition, self-reflection, and the potential of LLM in solving complex problems.', language='English', keywords='LLM, autonomous agents, planning, memory, tool use, task decomposition, self-reflection, challenges, case studies')

In [35]:
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str
    author: Optional[str]


class Info(BaseModel):
    """Information to extract"""
    papers: List[Paper]

In [36]:
extraction_model = model.bind_tools(
    [Info],
    tool_choice={"type": "function", "function": {"name": "Info"}}
)
extraction_chain = prompt | extraction_model | PydanticToolsParser(tools=[Info], first_tool_only=True) | (lambda x: x.papers)

In [37]:
extraction_chain.invoke({"input": page_content})

[Paper(title='LLM Powered Autonomous Agents', author='Lilian Weng')]

In [38]:
template = """A article will be passed to you. Extract from it all papers that are mentioned by this article follow by its author.

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do not make up or guess ANY extra information. Only extract what exactly is in the text."""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")
])

In [39]:
extraction_chain = prompt | extraction_model | PydanticToolsParser(tools=[Info], first_tool_only=True) | (lambda x: x.papers)

In [40]:
extraction_chain.invoke({"input": page_content})

[Paper(title='Chain of thought (CoT; Wei et al. 2022)', author='Wei et al. 2022'),
 Paper(title='Tree of Thoughts (Yao et al. 2023)', author='Yao et al. 2023'),
 Paper(title='LLM+P (Liu et al. 2023)', author='Liu et al. 2023'),
 Paper(title='ReAct (Yao et al. 2023)', author='Yao et al. 2023'),
 Paper(title='Reflexion (Shinn & Labash 2023)', author='Shinn & Labash 2023'),
 Paper(title='Chain of Hindsight (CoH; Liu et al. 2023)', author='Liu et al. 2023'),
 Paper(title='Algorithm Distillation (AD; Laskin et al. 2023)', author='Laskin et al. 2023')]

In [41]:
extraction_chain.invoke({"input": "hi"})

[Paper(title='Paper A', author='Author A'),
 Paper(title='Paper B', author='Author B')]

In [42]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

In [43]:
splits = text_splitter.split_text(doc.page_content)

In [44]:
len(splits)

15

In [45]:
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [46]:
flatten([[1, 2], [3, 4]])

[1, 2, 3, 4]

In [47]:
print(splits[0])

LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent 

In [48]:
from langchain_core.runnables import RunnableLambda

In [49]:
prep = RunnableLambda(
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)

In [50]:
prep.invoke("hi")

[{'input': 'hi'}]

In [51]:
chain = prep | extraction_chain.map() | flatten

In [52]:
chain.invoke(doc.page_content)

[Paper(title='AutoGPT', author=None),
 Paper(title='GPT-Engineer', author=None),
 Paper(title='BabyAGI', author=None),
 Paper(title='Chain of thought', author='Wei et al. 2022'),
 Paper(title='Tree of Thoughts', author='Yao et al. 2023'),
 Paper(title='LLM+P', author='Liu et al. 2023'),
 Paper(title='ReAct', author='Yao et al. 2023'),
 Paper(title='Reflexion', author='Shinn & Labash 2023'),
 Paper(title='Chain of Hindsight (CoH)', author='Liu et al. 2023'),
 Paper(title='Algorithm Distillation (AD)', author='Laskin et al. 2023'),
 Paper(title='Miller 1956', author=None),
 Paper(title='Duan et al. 2017', author=None),
 Paper(title='Laskin et al. 2023', author=None),
 Paper(title='ann-benchmarks.com', author=None),
 Paper(title='MRKL (Karpas et al. 2022)', author='Karpas et al.'),
 Paper(title='TALM (Tool Augmented Language Models; Parisi et al. 2022)', author='Parisi et al.'),
 Paper(title='Toolformer (Schick et al. 2023)', author='Schick et al.'),
 Paper(title='HuggingGPT (Shen et al. 